# Species Conditioning -- Evaluation

Three questions:

1. Does conditioning help at all? D-Cond against **D-EW**, which it matches in everything but the conditioning channel.
2. Does it close the gap? D-Cond against **Model C**, the flat 24-class model that beat the two-head models on freshness.
3. Where does it help? Freshness accuracy **per species**, against the number of training images each species has.

Question 3 is the one that tests the hypothesis rather than just measuring the outcome: if conditioning works by telling the freshness head which species it is looking at, the gain should concentrate on species the model sees least.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Pronnnnnnn/fish-freshness-mtl.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

import sys
sys.path.append('/content/repo/04_Src')

In [ ]:
DATASET_ZIP = '/content/drive/MyDrive/fish-freshness-mtl/8_fish_3_freshness.zip'
DATASET_ROOT = '/content/data/8_fish_3_freshness'

COND_DIR = '/content/drive/MyDrive/fish-freshness-mtl/species_conditioning'
COND_CHECKPOINTS = f'{COND_DIR}/checkpoints'
COND_RESULTS = f'{COND_DIR}/results'

MAIN_CHECKPOINTS = '/content/drive/MyDrive/fish-freshness-mtl/checkpoints'
MAIN_RESULTS = '/content/drive/MyDrive/fish-freshness-mtl/results/test_results_long.csv'

import os
os.makedirs('/content/data', exist_ok=True)
!unzip -q -n "$DATASET_ZIP" -d /content/data

In [ ]:
import numpy as np
import pandas as pd
import torch

from models import SpeciesConditionedMTL
from split_utils import load_split
from evaluate import (
    load_multitask_model, load_flat24_model,
    predict_multitask, predict_flat24,
    count_params, save_test_logits,
)
from metrics import evaluate_multitask
from comparison import to_long_format, paired_difference

train_df, val_df, test_df = load_split('/content/repo/02_Manifests/split_manifest.csv')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEEDS = [42, 43, 44]
len(test_df), DEVICE

## 1. Evaluate D-Cond

In [ ]:
records = []
cond_predictions = {}

for seed in SEEDS:
    run_name = f'ModelD_Cond_seed{seed}'
    model = load_multitask_model(f'{COND_CHECKPOINTS}/{run_name}.pt', DEVICE,
                                 model_factory=SpeciesConditionedMTL)
    sp_t, sp_p, fr_t, fr_p, logits = predict_multitask(model, test_df, DATASET_ROOT, DEVICE)
    cond_predictions[seed] = (sp_t, sp_p, fr_t, fr_p)
    save_test_logits(logits, run_name, COND_RESULTS)

    report = evaluate_multitask(sp_t, sp_p, fr_t, fr_p)
    records.append({'model': 'ModelD_Cond', 'seed': seed, 'tasks': {
        'species': report['species'],
        'freshness': report['freshness'],
        'joint': {'joint_accuracy': report['joint_accuracy']},
        'efficiency': {'params': count_params(model)},
    }})
    print(f"[{run_name}] freshness f1 {report['freshness']['f1_macro']:.4f} | "
          f"joint {report['joint_accuracy']:.4f}")

cond_long = to_long_format(records)
cond_long.to_csv(f'{COND_RESULTS}/dcond_results_long.csv', index=False)
cond_long.head()

## 2. Paired comparison against D-EW and Model C

`direction = b_better` means D-Cond won. As everywhere else, a configuration is only reported as ahead when the difference holds its sign on all three seeds.

In [ ]:
main_long = pd.read_csv(MAIN_RESULTS)
combined_long = pd.concat([main_long, cond_long], ignore_index=True)

COMPARISONS = [
    ('species', 'f1_macro'),
    ('freshness', 'f1_macro'),
    ('freshness', 'qwk'),
    ('joint', 'joint_accuracy'),
]

rows = []
for baseline in ['ModelD_EW', 'ModelC_flat24']:
    for task, metric in COMPARISONS:
        d = paired_difference(combined_long, baseline, 'ModelD_Cond', task, metric)
        d['per_seed_differences'] = str(d['per_seed_differences'])
        rows.append(d)

paired_df = pd.DataFrame(rows)
paired_df.to_csv(f'{COND_RESULTS}/dcond_paired_comparison.csv', index=False)
paired_df[['model_a', 'model_b', 'task', 'metric', 'mean_difference',
           'std_difference', 'consistent_sign', 'direction']].round(4)

## 3. Freshness accuracy per species

The hypothesis test. Conditioning should matter most where the model has least to go on, so the gain over D-EW should track inversely with how many training images a species has.

In [ ]:
def freshness_accuracy_per_species(species_true, freshness_true, freshness_pred):
    """Fraction of images of each species whose freshness is predicted correctly."""
    frame = pd.DataFrame({
        'species_idx': species_true,
        'correct': np.asarray(freshness_true) == np.asarray(freshness_pred),
    })
    return frame.groupby('species_idx')['correct'].mean()


def collect(model_name, loader, predict, checkpoint_dir, **load_kwargs):
    per_seed = []
    for seed in SEEDS:
        model = loader(f'{checkpoint_dir}/{model_name}_seed{seed}.pt', DEVICE, **load_kwargs)
        sp_t, _, fr_t, fr_p, _ = predict(model, test_df, DATASET_ROOT, DEVICE)
        per_seed.append(freshness_accuracy_per_species(sp_t, fr_t, fr_p))
    return pd.concat(per_seed, axis=1).mean(axis=1)


accuracy_by_model = {
    'ModelD_Cond': pd.concat(
        [freshness_accuracy_per_species(sp_t, fr_t, fr_p)
         for sp_t, _, fr_t, fr_p in cond_predictions.values()], axis=1).mean(axis=1),
    'ModelD_EW': collect('ModelD_EW', load_multitask_model, predict_multitask, MAIN_CHECKPOINTS),
    'ModelC_flat24': collect('ModelC_flat24', load_flat24_model, predict_flat24, MAIN_CHECKPOINTS),
}

per_species = pd.DataFrame(accuracy_by_model)
per_species.index.name = 'species_idx'
per_species.head()

In [ ]:
species_names = (train_df[['species_idx', 'species']]
                 .drop_duplicates().set_index('species_idx')['species'])

per_species = per_species.join(species_names)
per_species['n_train'] = train_df.groupby('species_idx').size()
per_species['n_test'] = test_df.groupby('species_idx').size()
per_species['cond_minus_ew'] = per_species['ModelD_Cond'] - per_species['ModelD_EW']
per_species['cond_minus_flat24'] = per_species['ModelD_Cond'] - per_species['ModelC_flat24']

per_species = per_species[['species', 'n_train', 'n_test', 'ModelD_EW', 'ModelC_flat24',
                           'ModelD_Cond', 'cond_minus_ew', 'cond_minus_flat24']]
per_species.sort_values('n_train').to_csv(f'{COND_RESULTS}/dcond_per_species.csv')
per_species.sort_values('n_train').round(4)

Sorted by training-set size, smallest first. If the hypothesis holds, `cond_minus_ew` should be largest at the top of this table and fade toward the bottom. A flat or noisy column means conditioning is not working the way the hypothesis says, whatever the overall averages do.

In [ ]:
correlation = per_species['n_train'].corr(per_species['cond_minus_ew'], method='spearman')
print(f'Spearman correlation, training images vs gain over D-EW: {correlation:.3f}')
print('Negative would support the hypothesis: bigger gains where there are fewer images.')
print('With 8 species this is descriptive only -- do not read it as a test.')